<a href="https://colab.research.google.com/github/creative-h/noiseFilterYT/blob/main/colab_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# YouTube Playlist Processing Pipeline - Google Colab

This notebook helps you run the YouTube playlist processing pipeline on Google Colab with GPU acceleration for faster transcription and audio processing.

## Features
- Download entire YouTube playlists
- Convert to WAV for high-quality processing
- Remove noise using AI-based or FFmpeg methods
- Optional audio super-resolution
- Normalize audio volume (EBU R128)
- Export high-quality MP3
- Transcribe using Faster-Whisper (GPU accelerated, optional)
- Summarize using OpenAI GPT (optional)

## Key Changes in Refactored Version
- **WAV intermediate format**: Avoids repeated lossy encoding
- **Lazy model loading**: AI models only load when needed (fixes RecursionError)
- **Per-video error isolation**: One failure doesn't stop the playlist
- **Optional transcription**: Disabled by default, audio processing works independently
- **Progress tracking**: Clear console output with [N/Total] format
- **Resume capability**: Skips existing files automatically

## Step 1: Check GPU Availability

First, let's check if a GPU is available for faster processing.

In [4]:
!nvidia-smi

Wed Jul  8 15:14:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Step 2: Clone the Repository

In [ ]:
!git clone https://github.com/creative-h/noiseFilterYT.git
%cd noiseFilterYT

# Pull latest changes in case you're re-running
!git pull origin main

# Copy the original config.py as backup (it has YDLP_COOKIES_FILE)
!cp config.py config_backup.py

Cloning into 'noiseFilterYT'...
remote: Enumerating objects: 24, done.
remote: Counting objects: 100% (24/24), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 24 (delta 8), reused 21 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (24/24), 27.16 KiB | 3.39 MiB/s, done.
Resolving deltas: 100% (8/8), done.
/content/noiseFilterYT


## Step 3: Install Dependencies

This will install all required packages including FFmpeg, yt-dlp, DeepFilterNet, and Faster-Whisper.

In [6]:
# Install FFmpeg (already installed from previous run)
!apt-get update -qq
!apt-get install -y ffmpeg

# Update pip and numpy to fix compatibility issues
!pip install -q --upgrade pip
!pip install -q --upgrade numpy

# Install CUDA-compatible PyTorch for GPU acceleration first,
# as it has strict setuptools requirements.
!pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu118

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 340 not upgraded.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 40.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cupy-cuda12x 13.3.0 requires numpy<2.3,>=1.22, but you have numpy 2.4.6 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.6 which is incompatible.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 2.4.6 which is incompatible.
tensorflow 2.18.0

In [7]:
# Install remaining Python dependencies
# Upgrade numpy first to avoid compatibility issues
!pip install -q --upgrade numpy packaging

# Install yt-dlp, faster-whisper, and openai
!pip install -q yt-dlp faster-whisper openai

# deepfilternet installation is optional and not needed if SKIP_DENOISE is True
# Uncomment the line below if you want to use denoising
# !pip install -q deepfilternet

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-core 0.3.68 requires packaging<25,>=23.2, but you have packaging 26.2 which is incompatible.
tensorflow 2.18.0 requires numpy<2.1.0,>=1.26.0, but you have numpy 2.4.6 which is incompatible.


In [ ]:
# Note: deepfilternet installation is skipped due to dependency conflicts
# If you need denoising, set SKIP_DENOISE=True in the settings below
# The pipeline will work without denoising for transcription

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
db-dtypes 1.4.3 requires packaging>=24.2.0, but you have packaging 23.2 which is incompatible.
google-cloud-bigquery 3.34.0 requires packaging>=24.2.0, but you have packaging 23.2 which is incompatible.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.


## Step 4: Configure YouTube Cookies (Optional but Recommended)

YouTube may require authentication for some videos. If you get "Sign in to confirm you're not a bot" errors, you need to provide cookies.

In [ ]:
### Option 1: Upload cookies.txt file

If you have a cookies.txt file from your browser, upload it here.

No API key set - summarization will be skipped


from google.colab import files
import os

# Upload cookies.txt file
print("Please upload your cookies.txt file if you have one.")
print("If you don't have one, skip this step and the pipeline will try without cookies.")
uploaded = files.upload()

# Check if cookies.txt was uploaded
if 'cookies.txt' in uploaded:
    print("cookies.txt uploaded successfully!")
    os.environ["YDLP_COOKIES_FILE"] = "cookies.txt"
else:
    print("No cookies.txt uploaded. Will try downloading without cookies.")
    print("Note: Some videos may require cookies to download.")

In [ ]:
### Option 2: How to get cookies.txt (if you don't have one)

1. Install the "Get cookies.txt" extension in your browser (Chrome/Firefox)
2. Go to YouTube and log in to your account
3. Click the extension icon and export cookies.txt
4. Upload the file using the cell above

**Note**: Cookies are only used for authentication with YouTube. They are not stored or shared.

Playlist URL: https://youtube.com/playlist?list=PLkTvpexFOP4AUCiBHZN9-DMzAtScY592t&si=9i-oZICQdVH-zRIs
Whisper Model: large-v3
Device: auto


In [ ]:
## Step 5: Configure OpenAI API (Optional)

If you want to use the summarization feature, set your OpenAI API key here.

import os

# Enter your OpenAI API key here (or leave empty to skip summarization)
OPENAI_API_KEY = ""  # @param {type:"string"}

if OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
    print("OpenAI API key set")
else:
    print("No API key set - summarization will be skipped")

In [ ]:
## Step 6: Configure Pipeline Settings

Adjust these settings based on your needs.

Configuration updated


# Playlist URL
PLAYLIST_URL = "https://youtube.com/playlist?list=PLkTvpexFOP4AUCiBHZN9-DMzAtScY592t&si=9i-oZICQdVH-zRIs"  # @param {type:"string"}

# Processing options
START_INDEX = 2  # @param {type:"integer"}
END_INDEX = 3  # @param {type:"integer"}

# Skip steps (set to True to skip)
SKIP_DOWNLOAD = False  # @param {type:"boolean"} - Set to True if files are already downloaded
SKIP_DENOISE = False  # @param {type:"boolean"}
SKIP_ENHANCE = False  # @param {type:"boolean"} - Skip audio enhancement
SKIP_NORMALIZE = False  # @param {type:"boolean"}
SKIP_TRANSCRIBE = False  # @param {type:"boolean"}
SKIP_SUMMARIZE = False  # @param {type:"boolean"}

# Feature enable/disable
ENABLE_TRANSCRIBE = False  # @param {type:"boolean"} - Enable transcription (disabled by default)
ENABLE_SUMMARIZE = False  # @param {type:"boolean"} - Enable summarization
ENABLE_ENHANCE = False  # @param {type:"boolean"} - Enable audio super-resolution

# Whisper model (tiny, base, small, medium, large-v3, distil-large-v3)
# Use 'distil-large-v3' for faster transcription with good accuracy
WHISPER_MODEL = "distil-large-v3"  # @param ["tiny", "base", "small", "medium", "large-v3", "distil-large-v3"]

# Device (auto, cpu, cuda)
WHISPER_DEVICE = "auto"  # @param ["auto", "cpu", "cuda"]

print(f"Playlist URL: {PLAYLIST_URL}")
print(f"Whisper Model: {WHISPER_MODEL}")
print(f"Device: {WHISPER_DEVICE}")
print(f"Skip Download: {SKIP_DOWNLOAD}")
print(f"Enable Transcribe: {ENABLE_TRANSCRIBE}")

[youtube:tab] Extracting URL: https://youtube.com/playlist?list=PLkTvpexFOP4AUCiBHZN9-DMzAtScY592t&si=9i-oZICQdVH-zRIs
[youtube:tab] PLkTvpexFOP4AUCiBHZN9-DMzAtScY592t: Downloading webpage
[youtube:tab] PLkTvpexFOP4AUCiBHZN9-DMzAtScY592t: Redownloading playlist API JSON with unavailable videos
[download] Downloading playlist: सुबोधिनी स्वाध्याय _गोपीगीत
[youtube:tab] Playlist सुबोधिनी स्वाध्याय _गोपीगीत: Downloading 2 items of 58
[download] Downloading item 1 of 2
[youtube] Extracting URL: https://www.youtube.com/watch?v=59MrzMqDrVk
[youtube] 59MrzMqDrVk: Downloading webpage


[youtube] 59MrzMqDrVk: Downloading android vr player API JSON
[info] 59MrzMqDrVk: Downloading 1 format(s): 251
[download] Destination: /content/raw/002-सुबोधिनी स्वाध्याय _गोपीगीत_001 (Subodhini_Gopigeet).webm
[download] 100% of   18.41MiB in 00:00:00 at 54.45MiB/s  
[ExtractAudio] Destination: /content/raw/002-सुबोधिनी स्वाध्याय _गोपीगीत_001 (Subodhini_Gopigeet).mp3
Deleting original file /content/raw/002-सुबोधिनी स्वाध्याय _गोपीगीत_001 (Subodhini_Gopigeet).webm (pass -k to keep)
[download] Downloading item 2 of 2
[youtube] Extracting URL: https://www.youtube.com/watch?v=GGuyyNh1LqA
[youtube] GGuyyNh1LqA: Downloading webpage


[youtube] GGuyyNh1LqA: Downloading android vr player API JSON
[info] GGuyyNh1LqA: Downloading 1 format(s): 251
[download] Destination: /content/raw/003-सुबोधिनी स्वाध्याय _गोपीगीत_002.webm
[download] 100% of   30.27MiB in 00:00:01 at 23.08MiB/s  
[ExtractAudio] Destination: /content/raw/003-सुबोधिनी स्वाध्याय _गोपीगीत_002.mp3
Deleting original file /content/raw/003-सुबोधिनी स्वाध्याय _गोपीगीत_002.webm (pass -k to keep)
[download] Finished downloading playlist: सुबोधिनी स्वाध्याय _गोपीगीत


## Step 7: Update Configuration

In [ ]:
# Update config.py with our settings
config_content = '''
"""
Configuration settings for the YouTube playlist processing pipeline.
"""

import os
from pathlib import Path

# Base directory
BASE_DIR = Path(__file__).parent

# Directory structure (refactored for proper workflow)
DOWNLOADS_DIR = BASE_DIR / "downloads"
WORKING_DIR = BASE_DIR / "working"
WAV_DIR = WORKING_DIR / "wav"
DENOISED_DIR = WORKING_DIR / "denoised"
ENHANCED_DIR = WORKING_DIR / "enhanced"
OUTPUT_DIR = BASE_DIR / "output"
FINAL_DIR = OUTPUT_DIR / "final"
TRANSCRIPT_DIR = OUTPUT_DIR / "transcripts"
LOGS_DIR = BASE_DIR / "logs"

# Legacy directories (for backward compatibility)
RAW_DIR = DOWNLOADS_DIR
MP3_DIR = DOWNLOADS_DIR
CLEAN_DIR = FINAL_DIR
SUMMARY_DIR = OUTPUT_DIR / "summaries"

# Create directories if they don't exist
for dir_path in [
    DOWNLOADS_DIR, WORKING_DIR, WAV_DIR, DENOISED_DIR, ENHANCED_DIR,
    OUTPUT_DIR, FINAL_DIR, TRANSCRIPT_DIR, SUMMARY_DIR, LOGS_DIR
]:
    dir_path.mkdir(parents=True, exist_ok=True)

# yt-dlp settings
YDLP_FORMAT = "bestaudio/best"
YDLP_COOKIES_FILE = os.getenv("YDLP_COOKIES_FILE", "cookies.txt")
YDLP_POSTPROCESSORS = []  # Disabled - we'll convert to WAV manually

# Audio processing settings
AUDIO_BITRATE = "256k"  # High quality MP3
AUDIO_SAMPLE_RATE = 48000  # Higher sample rate for better quality
WAV_BIT_DEPTH = 16  # 16-bit or 32-bit float

# FFmpeg loudnorm settings (EBU R128)
LOUDNORM_SETTINGS = {
    "i": "-16",  # Integrated loudness target (LUFS)
    "tp": "-1.5",  # True peak (dBTP)
    "lra": "11",  # Loudness range (LU)
}

# Noise suppression settings
NOISE_SUPPRESSION_ENABLED = True
NOISE_SUPPRESSION_STRENGTH = 0.5  # 0.0 to 1.0
NOISE_SUPPRESSION_METHOD = "ffmpeg"  # "ffmpeg" or "deepfilter" or "openvino"

# DeepFilterNet settings
DEEPFILTER_MODEL = "deepfilter_net"

# Audio super resolution settings
SUPER_RESOLUTION_ENABLED = False
SUPER_RESOLUTION_MODEL = "basic"  # Model to use for enhancement

# Whisper settings
TRANSCRIPTION_ENABLED = False  # CRITICAL: Disabled by default to avoid RecursionError
WHISPER_MODEL = "''' + WHISPER_MODEL + '''"
WHISPER_DEVICE = "''' + WHISPER_DEVICE + '''"
WHISPER_COMPUTE_TYPE = "int8"  # int8 for CPU compatibility, float16 for GPU

# LLM settings for summarization
SUMMARIZATION_ENABLED = False
LLM_API_KEY = os.getenv("OPENAI_API_KEY", "")
LLM_MODEL = "gpt-4o"
LLM_TEMPERATURE = 0.7
LLM_MAX_TOKENS = 2000

# Processing options
SKIP_EXISTING = True  # Skip files that already exist
DELETE_INTERMEDIATE = False  # Delete WAV files after processing
REMOVE_SILENCE = False
SILENCE_THRESHOLD = "-50dB"
MIN_SILENCE_DURATION = "1"

# Logging settings
LOG_LEVEL = "INFO"
LOG_FORMAT = "%(asctime)s - %(name)s - %(levelname)s - %(message)s"
LOG_FILE = LOGS_DIR / "pipeline.log"
CONSOLE_LOG_LEVEL = "INFO"  # Separate console log level
'''

with open('config.py', 'w') as f:
    f.write(config_content)

print("Configuration updated")

=== Processed Files ===

Raw audio files: 0
MP3 files: 0
Clean audio files: 0
Transcripts: 0
Summaries: 0


## Step 8: Ensure Configuration is Up-to-Date

This step ensures the config.py has all necessary settings including YDLP_COOKIES_FILE.

In [ ]:
# Check if config.py has YDLP_COOKIES_FILE, if not restore from backup
with open('config.py', 'r') as f:
    config_content = f.read()
    
if 'YDLP_COOKIES_FILE' not in config_content:
    print("YDLP_COOKIES_FILE not found in config.py, restoring from backup...")
    import shutil
    from pathlib import Path
    if Path('config_backup.py').exists():
        shutil.copy('config_backup.py', 'config.py')
        print("Restored config.py from backup")
    else:
        print("No backup found, adding YDLP_COOKIES_FILE manually...")
        # Add it manually
        with open('config.py', 'a') as f:
            f.write('\nYDLP_COOKIES_FILE = os.getenv("YDLP_COOKIES_FILE", "cookies.txt")\n')
        print("Added YDLP_COOKIES_FILE to config.py")
else:
    print("config.py already has YDLP_COOKIES_FILE")

## Step 9: Run the Pipeline

This will download and process the entire playlist (or the specified range).

In [ ]:
from main import Pipeline

# Initialize pipeline with feature flags
pipeline = Pipeline(
    noise_suppression_enabled=not SKIP_DENOISE,
    super_resolution_enabled=ENABLE_ENHANCE,
    transcription_enabled=ENABLE_TRANSCRIBE,
    summarization_enabled=ENABLE_SUMMARIZE,
)

# Run full pipeline
pipeline.run_full_pipeline(
    playlist_url=PLAYLIST_URL,
    start=START_INDEX if START_INDEX > 0 else None,
    end=END_INDEX if END_INDEX and END_INDEX > 0 else None,
    skip_download=SKIP_DOWNLOAD,
    skip_denoise=SKIP_DENOISE,
    skip_enhance=SKIP_ENHANCE,
    skip_normalize=SKIP_NORMALIZE,
    skip_transcribe=SKIP_TRANSCRIBE,
    skip_summarize=SKIP_SUMMARIZE,
)

## Step 10: Download Results

After processing, you can download the results.

In [ ]:
# List all processed files
from pathlib import Path

print("=== Processed Files ===")
print(f"\nDownloaded files: {len(list(Path('downloads').glob('*')))}")
print(f"WAV files: {len(list(Path('working/wav').glob('*.wav')))}")
print(f"Denoised files: {len(list(Path('working/denoised').glob('*.wav')))}")
print(f"Enhanced files: {len(list(Path('working/enhanced').glob('*.wav')))}")
print(f"Final MP3 files: {len(list(Path('output/final').glob('*.mp3')))}")
print(f"Transcripts: {len(list(Path('output/transcripts').glob('*.txt')))}")
print(f"Summaries: {len(list(Path('output/summaries').glob('*.md')))}")

NameError: name 'pipeline' is not defined

### Download Individual Files

Run this cell to see download links for all processed files.

In [ ]:
from google.colab import files

# Download all transcripts
transcript_files = list(Path('output/transcripts').glob('*.txt'))
if transcript_files:
    print(f"Downloading {len(transcript_files)} transcript files...")
    for f in transcript_files:
        files.download(f)
else:
    print("No transcript files found")

# Create ZIP of all results
!zip -r results.zip output/final/ output/transcripts/ output/summaries/

# Download ZIP
from google.colab import files
files.download('results.zip')

## Step 11: Alternative: Process Single Video

If you want to process just one video instead of a playlist.

In [ ]:
# Single video URL
VIDEO_URL = "https://youtu.be/bvhOsgRsAtg?si=S59HNuYh-6G9Dkod"  # @param {type:"string"}

if VIDEO_URL:
    pipeline.process_single_video(
        video_url=VIDEO_URL,
        skip_denoise=SKIP_DENOISE,
        skip_enhance=SKIP_ENHANCE,
        skip_normalize=SKIP_NORMALIZE,
        skip_transcribe=SKIP_TRANSCRIBE,
        skip_summarize=SKIP_SUMMARIZE,
    )
else:
    print("Please enter a video URL")

## Tips for Large Playlists

1. **Process in chunks**: Use START_INDEX and END_INDEX to process playlists in batches
2. **Use faster model**: Set WHISPER_MODEL to "distil-large-v3" for faster transcription
3. **Skip steps**: If you only need transcripts, set SKIP_DENOISE and SKIP_NORMALIZE to True
4. **Save to Drive**: Mount Google Drive to save results permanently

### Mount Google Drive (Optional)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Copy results to Drive
!cp -r clean/ /content/drive/MyDrive/youtube_pipeline/
!cp -r transcript/ /content/drive/MyDrive/youtube_pipeline/
!cp -r summary/ /content/drive/MyDrive/youtube_pipeline/

## Troubleshooting

- **"Sign in to confirm you're not a bot" error**: Upload cookies.txt file in Step 4
- **Out of memory**: Use a smaller Whisper model ("medium" or "small")
- **Slow processing**: Ensure GPU is enabled in Colab (Runtime > Change runtime type > GPU)
- **Download errors**: Check playlist URL is public and accessible
- **DeepFilterNet errors**: Skip denoising with SKIP_DENOISE=True